<a href="https://colab.research.google.com/github/geekabhi/AlgorithmPractice/blob/master/rag_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Workshop - Build a Docs Chatbot from Scratch

**Goal:** By the end of this notebook you will understand what RAG is, why it matters, and you'll have a working chatbot that answers questions about [Claude Code](https://docs.claude.com/en/docs/claude-code) docs - with citations.

---

## What is RAG?

**RAG = Retrieval-Augmented Generation**

Think of it like an open-book exam vs. a closed-book exam:

|    INTUITION               | Closed-book (plain LLM)                       | Open-book (RAG) |
|-------------------|-----------------------------------------------|-----------------|
| **Knowledge**     | Only what the model memorized during training | Model can look up facts from YOUR documents |
| **Freshness**     | Frozen at training cutoff date                | Always up-to-date (uses live docs) |
| **Hallucination** | Makes up plausible-sounding answers           | Grounded in real text — with citations |
| **Custom data**   | Knows nothing about your internal docs        | Can search your PDFs, wikis, code, etc. |

### The RAG pipeline in 4 steps

```
┌─────────────-┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  1. LOAD &   │     │ 2. EMBED &  │     │ 3. RETRIEVE │     │ 4. GENERATE │
│    CHUNK     │────▶│    STORE    │────▶│  (search)   │────▶│  (answer)   │
│              │     │             │     │             │     │             │
│ Parse docs   │     │ Convert to  │     │ Find chunks │     │ LLM reads   │
│ into small   │     │ vectors &   │     │ most similar│     │ the chunks  │
│ pieces       │     │ store in DB │     │ to the      │     │ and writes  │
│ ("chunks")   │     │             │     │ user query  │     │ an answer   │
└─────────────-┘     └─────────────┘     └─────────────┘     └─────────────┘
     OFFLINE              OFFLINE            ONLINE              ONLINE
  (done once)          (done once)      (every question)    (every question)
```

### Why not just paste the whole document into the LLM?

1. **Context window limits:** LLMs can only read so much text at once (e.g. 128K tokens ≈ a 300-page book). Your docs might be bigger.
2. **Cost:** Sending 300 pages per question is expensive. Sending just the 5 relevant paragraphs is cheap.
3. **Accuracy:** less noise = better answers. The model focuses on what matters.

---

### What we'll use

| Component | Tool | Why |
|---|---|---|
| Document loading | **LangChain** (`WebBaseLoader`, `PyPDFLoader`) | Industry-standard, supports 100+ source types |
| Chunking | **LangChain** (`RecursiveCharacterTextSplitter`) | Battle-tested, handles text splitting intelligently |
| Embeddings | **OpenAI** (`text-embedding-3-small`) | Cheap, fast, high quality |
| Vector store | **ChromaDB** | Zero-config, runs locally, great for prototyping |
| LLM | **OpenAI** (`gpt-4o-mini`) | Fast and cheap for generation |
| UI | **Gradio** | One-line chat UI that works in notebooks |

Let's build it step by step.

---
## Step 0: Install & Setup

We install a few Python packages. If you've used `pip install` before, this is the same thing.

| Package | What it does |
|---|---|
| `langchain` | Framework for building LLM apps - we use its document loaders and text splitters |
| `langchain-community` | Community-contributed loaders (PDF, web, etc.) |
| `langchain-text-splitters` | The chunking utilities |
| `openai` | Python client for the OpenAI API (embeddings + chat) |
| `chromadb` | Local vector database - stores and searches embeddings |
| `pypdf` | PDF parsing library (used by LangChain's PDF loader under the hood) |
| `gradio` | Instant web UI for demos |
| `httpx`, `tqdm` | HTTP requests and progress bars |

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters openai chromadb pypdf pymupdf httpx tqdm gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/20

In [ ]:
import os

# In Google Colab: click the key icon in the left sidebar,
# add a secret named OPENAI_API_KEY, toggle "Notebook access" on.
# Running locally? Just set the env var: export OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from Colab secrets.")
except Exception:
    assert os.environ.get("OPENAI_API_KEY"), (
        "Set OPENAI_API_KEY in your environment or Colab secrets."
    )
    print("OPENAI_API_KEY found in environment.")

Loaded OPENAI_API_KEY from Colab secrets.


---
## Step 1: Load Documents

Before we can search anything, we need to **load** the raw text. In real projects your data might come from:
- Web pages / documentation sites
- PDF files (contracts, manuals, reports)
- Databases, Confluence, Notion, Google Docs, etc.

LangChain provides **Document Loaders** for all of these. A loader returns a list of `Document` objects, each with:
- `.page_content`: the text
- `.metadata`: info like source URL, page number, title

We'll demo **two** loaders:
1. **Web loader:** fetch the Claude Code docs from a URL
2. **PDF loader:** parse a local PDF file

### 1a. Loading from the web

Anthropic publishes all Claude Code docs in a single text file. We'll fetch it and split it into per-page documents.

In [ ]:
import re
import httpx
from langchain_core.documents import Document

# -- Fetch the raw docs --
URL = "https://code.claude.com/docs/llms-full.txt"
print(f"Fetching {URL} ...")
raw_text = httpx.get(URL, timeout=60, follow_redirects=True).raise_for_status().text
print(f"Downloaded {len(raw_text):,} characters")

# -- Split into per-page Documents --
# The file has headers like:  # Page Title\nSource: https://...\n
# We split on those headers to get one Document per docs page.
header_pattern = re.compile(r"^# (.+?)\nSource: (\S+)\s*\n", flags=re.MULTILINE)
matches = list(header_pattern.finditer(raw_text))

web_docs = []
for i, m in enumerate(matches):
    body_start = m.end()
    body_end = matches[i + 1].start() if i + 1 < len(matches) else len(raw_text)
    body = raw_text[body_start:body_end].strip()
    if body:
        web_docs.append(Document(
            page_content=f"# {m.group(1).strip()}\n\n{body}",
            metadata={"title": m.group(1).strip(), "source": m.group(2).strip()}
        ))

print(f"\nParsed into {len(web_docs)} documents (one per docs page).")
print(f"\nExample :- first doc title: '{web_docs[0].metadata['title']}'")
print(f"First 200 chars: {web_docs[0].page_content[:200]}...")

Fetching https://code.claude.com/docs/llms-full.txt ...
Downloaded 4,038,538 characters

Parsed into 140 documents (one per docs page).

Example — first doc title: 'Set up Claude Code for your organization'
First 200 chars: # Set up Claude Code for your organization

A decision map for administrators deploying Claude Code, covering API providers, managed settings, policy enforcement, usage monitoring, and data handling.
...


In [ ]:
len(web_docs)

140

### 1b. Loading from a PDF file

This is one of the most common real-world use cases: you have internal PDFs (SOPs, contracts, manuals) and you want an AI that can answer questions about them.

LangChain's `PyPDFLoader` makes this a one-liner. It:
1. Opens the PDF file
2. Extracts text from each page
3. Returns a `Document` per page with `page_number` in the metadata

**Try it yourself:** Upload any PDF to this notebook and change the path below.

> If you don't have a PDF handy, we'll download a sample one.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# -- Download a sample PDF if you don't have one --
SAMPLE_PDF_URL = "https://arxiv.org/pdf/2005.11401"  # The original RAG paper!
PDF_PATH = "./sample_rag_paper.pdf"

if not os.path.exists(PDF_PATH):
    print(f"Downloading sample PDF from {SAMPLE_PDF_URL} ...")
    resp = httpx.get(SAMPLE_PDF_URL, timeout=60, follow_redirects=True)
    resp.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Saved to {PDF_PATH} ({len(resp.content):,} bytes)")

# -- Load the PDF --
pdf_loader = PyPDFLoader(PDF_PATH)
pdf_docs_1 = pdf_loader.load()

print(f"\nLoaded {len(pdf_docs_1)} pages from the PDF.")
print(f"\nPage 1 metadata: {pdf_docs_1[0].metadata}")
print(f"Page 1 text (first 300 chars):\n{pdf_docs_1[0].page_content[:300]}...")


Loaded 19 pages from the PDF.

Page 1 metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': './sample_rag_paper.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}
Page 1 text (first 300 chars):
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University C...


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(
    file_path = "/content/Fast transformer decoding- one write head is all you need.pdf"
    )

pdf_docs = loader.load()


print(f"\nLoaded {len(pdf_docs)} pages from the PDF.")
print(f"\nPage 1 metadata: {pdf_docs[0].metadata}")
print(f"Page 1 text (first 300 chars):\n{pdf_docs[0].page_content[:600]}...")


Loaded 9 pages from the PDF.

Page 1 metadata: {'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.22', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-11-06T20:56:43-05:00', 'source': '/content/Fast transformer decoding- one write head is all you need.pdf', 'file_path': '/content/Fast transformer decoding- one write head is all you need.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2019-11-06T20:56:43-05:00', 'trapped': '', 'modDate': "D:20191106205643-05'00'", 'creationDate': "D:20191106205643-05'00'", 'page': 0}
Page 1 text (first 300 chars):
arXiv:1911.02150v1  [cs.NE]  6 Nov 2019
Fast Transformer Decoding: One Write-Head is All
You Need
Noam Shazeer
Google
noam@google.com
November 7, 2019
Abstract
Multi-head attention layers, as used in the Transformer neural sequence model, are a powerful alter-
native to RNNs for moving information across and between sequences. While training these layers is

### Quick comparison: what a Document looks like

Whether it came from a URL or a PDF, LangChain gives us the same `Document` object. This is powerful: the rest of our pipeline doesn't care where the data came from.

In [ ]:
print("=== Web Document ===")
print(f"  Type:     {type(web_docs[0]).__name__}")
print(f"  Metadata: {web_docs[0].metadata}")
print(f"  Content:  {web_docs[0].page_content[:200]}...")

print("\n=== PDF Document ===")
print(f"  Type:     {type(pdf_docs[0]).__name__}")
print(f"  Metadata: {pdf_docs[0].metadata}")
print(f"  Content:  {pdf_docs[0].page_content[:200]}...")

print("\n>> Same Document type! The rest of the pipeline is identical.")

=== Web Document ===
  Type:     Document
  Metadata: {'title': 'Set up Claude Code for your organization', 'source': 'https://code.claude.com/docs/en/admin-setup'}
  Content:  # Set up Claude Code for your organization

A decision map for administrators deploying Claude Code, covering API providers, managed settings, policy enforcement, usage monitoring, and data handling.
...

=== PDF Document ===
  Type:     Document
  Metadata: {'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.22', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-11-06T20:56:43-05:00', 'source': '/content/Fast transformer decoding- one write head is all you need.pdf', 'file_path': '/content/Fast transformer decoding- one write head is all you need.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2019-11-06T20:56:43-05:00', 'trapped': '', 'modDate': "D:20191106205643-05'00'", 'creationDate': "D:20191106205643-05'00'", 'page': 0}
  

---
## Step 2: Chunk the Documents

### Why do we need to chunk?

Imagine you're searching a textbook for the answer to a question. You wouldn't hand someone the *entire* book and say "the answer is in here somewhere." You'd point them to the right **paragraph** or **section**.

Chunking is the same idea:
- **Too big** (whole pages): retrieval is imprecise, LLM gets distracted by irrelevant text
- **Too small** (individual sentences): loses context, the chunk might not make sense alone
- **Just right** (~500-1500 chars): captures a complete thought, precise enough for retrieval

### LangChain's RecursiveCharacterTextSplitter

Instead of building our own chunker, we use LangChain's `RecursiveCharacterTextSplitter`. It's the industry default because it's smart about where it cuts:


Two key parameters:
- **`chunk_size`:** maximum characters per chunk (we use 512)
- **`chunk_overlap`:** how many characters overlap between adjacent chunks (we use 50)

**Why overlap?** If a sentence gets split across two chunks, the overlap ensures the full sentence appears in at least one chunk.

```
Without overlap:   [chunk 1: "The config file is"] [chunk 2: "located in ~/.claude/settings.json"]
With overlap:      [chunk 1: "The config file is located in"] [chunk 2: "is located in ~/.claude/settings.json"]
                                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^ overlap ^
```

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -- Configure the splitter --
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,       # max characters per chunk
    chunk_overlap=50,     # overlap between adjacent chunks
)

# -- Chunk the web docs --
# split_documents() preserves the metadata from each source Document
web_chunks = splitter.split_documents(web_docs)

print(f"Web docs:  {len(web_docs)} documents  -->  {len(web_chunks)} chunks")
print(f"\nAvg chunk size: {sum(len(c.page_content) for c in web_chunks) // len(web_chunks)} chars")

# -- Show an example chunk --
example = web_chunks[len(web_chunks) // 2]
print(f"\n--- Example chunk ---")
print(f"Metadata: {example.metadata}")
print(f"Text ({len(example.page_content)} chars):")
print(example.page_content[:300] + "...")

Web docs:  140 documents  -->  11059 chunks

Avg chunk size: 345 chars

--- Example chunk ---
Metadata: {'title': 'Monitoring', 'source': 'https://code.claude.com/docs/en/monitoring-usage'}
Text (329 chars):
| `skill_name`      | Skill name for the Skill tool                                                                                      | `OTEL_LOG_TOOL_DETAILS` |
| `subagent_type`   | Subagent type for the Task tool                                                                                  ...


In [ ]:
# -- Chunk the PDF docs too (same splitter, same code!) --
pdf_chunks = splitter.split_documents(pdf_docs)
print(f"PDF docs:  {len(pdf_docs)} pages  -->  {len(pdf_chunks)} chunks")

# Show one
print(f"\n--- Example PDF chunk ---")
print(f"Metadata: {pdf_chunks[0].metadata}")
print(f"Text: {pdf_chunks[0].page_content[:110]}...")

PDF docs:  7 pages  -->  74 chunks

--- Example PDF chunk ---
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-24T01:50:57+00:00', 'source': '/content/Jasper and Stells- distillation of SOTA embedding models.pdf', 'file_path': '/content/Jasper and Stells- distillation of SOTA embedding models.pdf', 'total_pages': 7, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-01-24T01:50:57+00:00', 'trapped': '', 'modDate': 'D:20250124015057Z', 'creationDate': 'D:20250124015057Z', 'page': 0}
Text: Jasper and Stella: distillation of SOTA embedding models
Dun Zhang1, Jiacheng Li1∗, Ziyang Zeng1,2, Fulong Wan...


### Key takeaway

Notice how the **same splitter** works on both web docs and PDF docs. That's the power of LangChain's `Document` abstraction - load from *any* source, chunk with *one* tool.

For the rest of this notebook, we'll use the **web docs chunks** to build the chatbot (since the Claude Code docs are more useful for a demo). But you could swap in the PDF chunks or combine both with zero code changes.

---
## Step 3: Embed & Store in a Vector Database

### What is an embedding?

An **embedding** is just a list of numbers (a "vector") that captures the *meaning* of a piece of text.

```
"How do I install Python?"  -->  [0.021, -0.034, 0.087, ..., 0.012]   (1536 numbers)
"Python installation guide"  -->  [0.019, -0.031, 0.091, ..., 0.010]   (1536 numbers)
"Best pizza in New York"     -->  [0.854, 0.123, -0.445, ..., 0.667]   (1536 numbers)
```

The first two vectors are **close together** (similar meaning). The pizza one is **far away**.

### What is a vector database?

A vector database stores these vectors and lets you do **similarity search**: "given this query vector, find me the 5 closest stored vectors."

We use **ChromaDB** - it's a local, file-based vector DB. No server, no cloud account, no config. Perfect for prototyping.

```
┌──────────────────────────────────────────────────────────┐
│                    ChromaDB                              │
│                                                          │
│   chunk: "Install Claude Code with npm install -g..."    │
│   vector: [0.021, -0.034, 0.087, ...]                    │
│   metadata: {title: "Installation", source: "..."}       │
│                                                          │
│   chunk: "Hooks are shell commands that run on..."       │
│   vector: [0.145, 0.067, -0.023, ...]                    │
│   metadata: {title: "Hooks", source: "..."}              │
│                                                          │
│   ... hundreds more ...                                  │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from tqdm.auto import tqdm

# -- Config --
CHROMA_DIR = "./chroma_db"
COLLECTION = "claude_code_docs"
EMBED_MODEL = "text-embedding-3-small"
BATCH_SIZE = 100

# -- Create (or recreate) the Chroma collection --
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete existing collection if it exists (so this cell is re-runnable)
try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass

embed_fn = OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name=EMBED_MODEL,
)

collection = chroma_client.create_collection(
    name=COLLECTION,
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},  # index name: hnsw, use cosine similarity
)

# -- Add all chunks to the vector DB --
# We need IDs, documents (text), and metadata for each chunk.
print(f"Embedding and storing {len(web_chunks)} chunks...")
print(f"(This calls the OpenAI Embeddings API — takes ~30-60 seconds)\n")

for i in tqdm(range(0, len(web_chunks), BATCH_SIZE)):
    batch = web_chunks[i : i + BATCH_SIZE]
    collection.add(
        ids=[f"chunk-{i+j}" for j, _ in enumerate(batch)],
        documents=[c.page_content for c in batch],
        metadatas=[
            {
                "title": c.metadata.get("title", "Unknown"),
                "source": c.metadata.get("source", "Unknown"),
            }
            for c in batch
        ],
    )

print(f"\nDone! Collection has {collection.count()} embeddings stored.")

Embedding and storing 11059 chunks...
(This calls the OpenAI Embeddings API — takes ~30-60 seconds)



  0%|          | 0/111 [00:00<?, ?it/s]


Done! Collection has 11059 embeddings stored.


---
## Step 4: Retrieve (Search)

Now we can **search** our docs. This is pure vector similarity, no LLM involved yet.

How it works:
1. Your question gets embedded into a vector (same model as the chunks)
2. Chroma finds the top `k` stored vectors closest to your question vector
3. Returns the corresponding text chunks

**This is the most important step in RAG.** If retrieval returns the wrong chunks, no amount of clever prompting can save the answer. That's why we always inspect retrieval results before building the full pipeline.

**Analogy:** It's like a search engine, but instead of matching keywords, it matches *meaning*. "How do I set up hooks?" will find chunks about hooks even if the chunk text says "event listeners" instead of "hooks".

In [ ]:
TOP_K = 5  # How many chunks to retrieve per query

def retrieve(query, k=TOP_K):
    """Search the vector DB and return the top-k most relevant chunks."""
    return collection.query(query_texts=[query], n_results=k)

# -- Try a search --
test_query = "What permission modes are available?"
results = retrieve(test_query)

print(f"Query: \"{test_query}\"\n")
print(f"Top {TOP_K} results:\n")

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    similarity = 1 - dist  # Chroma returns distance; 1 - distance = similarity
    print(f"--- Rank {i}  (similarity: {similarity:.3f}) ---")
    print(f"Title:  {meta['title']}")
    print(f"Source: {meta['source']}")
    print(f"Text:   {doc[:250]}...")
    print()

Query: "What permission modes are available?"

Top 5 results:

--- Rank 1  (similarity: 0.745) ---
Title:  How the agent loop works
Source: https://code.claude.com/docs/en/agent-sdk/agent-loop
Text:   * **`permission_mode` / `permissionMode`** controls what happens to tools that aren't covered by allow or deny rules. See [Permission mode](#permission-mode) for available modes....

--- Rank 2  (similarity: 0.716) ---
Title:  Configure permissions
Source: https://code.claude.com/docs/en/agent-sdk/permissions
Text:   ## Permission modes

Permission modes provide global control over how Claude uses tools. You can set the permission mode when calling `query()` or change it dynamically during streaming sessions.

### Available modes

The SDK supports these permissio...

--- Rank 3  (similarity: 0.708) ---
Title:  Agent SDK reference - TypeScript
Source: https://code.claude.com/docs/en/agent-sdk/typescript
Text:   | `permissionMode`                  | [`PermissionMode`](#permissionmode)      

### Experiment!

Try changing the `test_query` above to different questions and re-run the cell. Good questions to try:
- "What is a CLAUDE.md file?"
- "How do I configure MCP servers?"
- "What permission modes are available?"

Do the top results make sense? If they don't, that's a retrieval problem and you'd fix it by improving your chunks, not your prompts.

---
## Step 5: Generate (The "G" in RAG)

Now we combine **retrieval** with **generation**:

1. Retrieve the top-k chunks for the user's question
2. Stuff them into the LLM's prompt as "context"
3. Ask the LLM to answer **using only that context**

The **system prompt** is critical. It tells the LLM:
- Only use the provided context (prevents hallucination)
- Say "I don't know" if the answer isn't there (prevents making things up)
- Cite source URLs (lets users verify the answer)

```
┌─────────────────────────────────────────────┐
│              LLM Prompt                     │
│                                             │
│  SYSTEM: You are a docs assistant.          │
│          Answer ONLY from context below.    │
│          Cite sources. Say IDK if unsure.   │
│                                             │
│  USER:   Context:                           │
│          [1] chunk about hooks ...          │
│          [2] chunk about hooks config ...   │
│          [3] chunk about events ...         │
│                                             │
│          Question: How do hooks work?       │
└─────────────────────────────────────────────┘
```

In [ ]:
from openai import OpenAI

CHAT_MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """You are a documentation assistant for Claude Code.
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I don't know based on the docs."
Always cite the source URLs of the chunks you used, like [1], [2].
Keep answers concise and code-accurate."""

oai = OpenAI()

# Augmentation step
def build_context(query, retrieved):
    """Format retrieved chunks into a numbered context block for the LLM."""
    docs = retrieved["documents"][0]
    metas = retrieved["metadatas"][0]
    blocks = [
        f"[{i}] Source: {m['source']}\n{d}"
        for i, (d, m) in enumerate(zip(docs, metas), start=1)
    ]
    context_str = "\n\n---\n\n".join(blocks)
    return f"Context:\n\n{context_str}\n\nQuestion: {query}", metas


def ask(query):
    """Full RAG pipeline: retrieve chunks, build prompt, call LLM."""
    retrieved = retrieve(query, k=TOP_K)
    user_msg, metas = build_context(query, retrieved)
    response = oai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.1,  # low temperature = more deterministic
    )
    return response.choices[0].message.content, metas


# -- Try it! --
question = "How do I create a custom slash command in Claude Code?"
answer, sources = ask(question)

print(f"Q: {question}\n")
print(f"A: {answer}")
print(f"\n--- Sources used ---")
for i, s in enumerate(sources, 1):
    print(f"  [{i}] {s['title']}  —  {s['source']}")

Q: How do I create a custom slash command in Claude Code?

A: To create a custom slash command in Claude Code, you need to define it as a markdown file in specific directories. The recommended format is to place the markdown file in the `.claude/skills/<name>/SKILL.md` directory. This format supports both slash-command invocation (`/name`) and autonomous invocation by Claude. 

For legacy support, you can also use the `.claude/commands/*.md` directory, but the newer format is preferred. 

For more details, refer to the documentation on creating custom slash commands [here](https://code.claude.com/docs/en/agent-sdk/slash-commands) [1][5].

--- Sources used ---
  [1] Slash Commands in the SDK  —  https://code.claude.com/docs/en/agent-sdk/slash-commands
  [2] Slash Commands in the SDK  —  https://code.claude.com/docs/en/agent-sdk/slash-commands
  [3] CLI reference  —  https://code.claude.com/docs/en/cli-reference
  [4] Agent SDK overview  —  https://code.claude.com/docs/en/agent-sdk/overv

In [ ]:
question = "What is a CLAUDE.md file?"
answer, sources = ask(question)

print(f"Q: {question}\n")
print(f"A: {answer}")
print(f"\n--- Sources used ---")
for i, s in enumerate(sources, 1):
    print(f"  [{i}] {s['title']}  —  {s['source']}")

Q: What is a CLAUDE.md file?

A: A CLAUDE.md file is a markdown file that contains persistent instructions for Claude, loaded at the start of every session. It can include project conventions, architecture notes, and workflow rules. The file is read in plain text and allows Claude to retain context across sessions. It survives compaction and is re-read fresh from disk afterward [1][4].

--- Sources used ---
  [1] Best practices for Claude Code  —  https://code.claude.com/docs/en/best-practices
  [2] How Claude remembers your project  —  https://code.claude.com/docs/en/memory
  [3] Configure permissions  —  https://code.claude.com/docs/en/permissions
  [4] Glossary  —  https://code.claude.com/docs/en/glossary
  [5] How Claude remembers your project  —  https://code.claude.com/docs/en/memory


---
## Step 6: Evaluate

### Why evaluate?

As engineers, you'd never ship code without tests. Same principle here - we need to measure how well our RAG system works before trusting it.

We measure two things:

| Metric | What it measures | How |
|---|---|---|
| **Retrieval hit-rate** | Did we find the right chunks? | Check if any top-k chunk comes from the expected source URL |
| **Answer quality** | Is the generated answer correct? | Use an LLM as a judge to score 1-5 against expected answers |

**Why both?** If retrieval is bad but answers are good, the LLM is getting lucky (or hallucinating). If retrieval is good but answers are bad, your prompt needs work. Both metrics together tell you *where* to focus improvements.

Our eval set is small (5 questions). I
n production you'd want 50-100+. But 5 is enough to catch major issues.

In [ ]:
# -- Evaluation dataset: hand-crafted question/answer pairs --
EVAL_SET = [
    {
        "question": "How do I create a custom slash command in Claude Code?",
        "expected_source_contains": "slash-commands",
        "expected_answer_gist": (
            "Create a markdown file in .claude/commands/ (or ~/.claude/commands/) "
            "whose filename is the command name; the file body becomes the prompt"
        ),
    },
    {
        "question": "What permission modes does Claude Code support?",
        "expected_source_contains": "permissions",
        "expected_answer_gist": (
            "Modes include default, acceptEdits, plan, and bypassPermissions; "
            "they control which tools can run without prompting"
        ),
    },
    {
        "question": "How do hooks work in Claude Code?",
        "expected_source_contains": "hooks",
        "expected_answer_gist": (
            "Hooks are shell commands in settings.json fired on events like "
            "PreToolUse / PostToolUse / Stop; they react to or block tool calls"
        ),
    },
    {
        "question": "How do I configure an MCP server in Claude Code?",
        "expected_source_contains": "mcp",
        "expected_answer_gist": (
            "Use `claude mcp add` or edit .mcp.json / settings.json; "
            "command, args, env vars; stdio / SSE / HTTP transports"
        ),
    },
    {
        "question": "What is a CLAUDE.md file used for?",
        "expected_source_contains": "memory",
        "expected_answer_gist": (
            "CLAUDE.md is a memory file loaded at session start; project-specific "
            "instructions; stacks across user/project/local scopes"
        ),
    },
]

# -- LLM-as-judge: score the generated answer against expected --
JUDGE_PROMPT = """You are scoring how well a generated answer matches an expected answer.

Question: {question}
Expected answer (gist): {expected}
Generated answer: {generated}

Score 1-5:
  1 = wrong or empty
  2 = partially relevant but missing key facts
  3 = roughly right but vague or incomplete
  4 = correct, covers the gist, minor omissions ok
  5 = correct, complete, and well cited

Reply with just the integer score, nothing else."""


def judge_answer(question, expected, generated):
    resp = oai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{
            "role": "user",
            "content": JUDGE_PROMPT.format(
                question=question, expected=expected, generated=generated
            ),
        }],
        temperature=0,
    )
    raw = (resp.choices[0].message.content or "").strip()
    for tok in raw.split():
        try:
            return max(1, min(5, int(tok)))
        except ValueError:
            continue
    return 1


# -- Run the eval --
print("Running evaluation...\n")
total_hits = 0
total_score = 0

for item in EVAL_SET:
    # Check retrieval
    r = retrieve(item["question"], k=TOP_K)
    sources = [m["source"] for m in r["metadatas"][0]]
    hit = any(item["expected_source_contains"] in s for s in sources)
    total_hits += int(hit)

    # Check answer quality
    answer, _ = ask(item["question"])
    score = judge_answer(item["question"], item["expected_answer_gist"], answer)
    total_score += score

    status = "HIT " if hit else "MISS"
    print(f"  [{status}] score={score}/5  {item['question']}")

n = len(EVAL_SET)
print(f"\n{'='*50}")
print(f"  Retrieval hit-rate:  {total_hits}/{n} = {total_hits/n:.0%}")
print(f"  Avg answer score:    {total_score/n:.1f} / 5.0")
print(f"{'='*50}")

if total_hits == n:
    print("\n  Retrieval is solid - all expected sources found.")
else:
    print(f"\n  {n - total_hits} retrieval miss(es) - consider adjusting chunk_size or TOP_K.")

if total_score / n >= 4:
    print("  Answer quality is good!")
else:
    print("  Answer quality needs improvement - review the system prompt or chunk size.")

Running evaluation...

  [HIT ] score=4/5  How do I create a custom slash command in Claude Code?
  [HIT ] score=3/5  What permission modes does Claude Code support?
  [HIT ] score=5/5  How do hooks work in Claude Code?
  [HIT ] score=4/5  How do I configure an MCP server in Claude Code?
  [HIT ] score=4/5  What is a CLAUDE.md file used for?

  Retrieval hit-rate:  5/5 = 100%
  Avg answer score:    4.0 / 5.0

  Retrieval is solid - all expected sources found.
  Answer quality is good!


**Check the quality of one question in the evals set:**

In [ ]:
item = EVAL_SET[1]
# Check retrieval
r = retrieve(item["question"], k=TOP_K)
for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    similarity = 1 - dist  # Chroma returns distance; 1 - distance = similarity
    print(f"--- Rank {i}  (similarity: {similarity:.3f}) ---")
    print(f"Title:  {meta['title']}")
    print(f"Source: {meta['source']}")
    print(f"Text:   {doc[:250]}...")
    print()
sources = [m["source"] for m in r["metadatas"][0]]
hit = any(item["expected_source_contains"] in s for s in sources)
total_hits += int(hit)

# Check answer quality
answer, _ = ask(item["question"])
print(f"Answer: ", answer)
score = judge_answer(item["question"], item["expected_answer_gist"], answer)
total_score += score

status = "HIT " if hit else "MISS"
print(f"  [{status}] score={score}/5  {item['question']}")

--- Rank 1  (similarity: 0.745) ---
Title:  How the agent loop works
Source: https://code.claude.com/docs/en/agent-sdk/agent-loop
Text:   * **`permission_mode` / `permissionMode`** controls what happens to tools that aren't covered by allow or deny rules. See [Permission mode](#permission-mode) for available modes....

--- Rank 2  (similarity: 0.716) ---
Title:  Configure permissions
Source: https://code.claude.com/docs/en/agent-sdk/permissions
Text:   ## Permission modes

Permission modes provide global control over how Claude uses tools. You can set the permission mode when calling `query()` or change it dynamically during streaming sessions.

### Available modes

The SDK supports these permissio...

--- Rank 3  (similarity: 0.708) ---
Title:  Agent SDK reference - TypeScript
Source: https://code.claude.com/docs/en/agent-sdk/typescript
Text:   | `permissionMode`                  | [`PermissionMode`](#permissionmode)                                                                     

---
## Step 7: Interactive Chatbot

Let's wrap everything in a simple chat UI using Gradio. This gives you a real chatbot you can interact with — type questions and get answers with sources.

This is the same pipeline as above (retrieve → build prompt → call LLM), just with a UI on top.

In [ ]:
import gradio as gr


def chatbot(message, history):
    """Handle a chat message: retrieve, generate, format with sources."""
    retrieved = retrieve(message, k=TOP_K)
    user_msg, metas = build_context(message, retrieved)

    resp = oai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.1,
    )
    answer = resp.choices[0].message.content

    # Append source list to the answer
    source_lines = [
        f"[{i}] {m['title']} — {m['source']}"
        for i, m in enumerate(metas, 1)
    ]
    return f"{answer}\n\n---\n**Sources retrieved:**\n" + "\n".join(source_lines)


gr.ChatInterface(
    fn=chatbot,
    title="Claude Code Docs Chatbot (RAG)",
    description=(
        "Ask anything about Claude Code. "
        "Powered by LangChain + ChromaDB + gpt-4o-mini."
    ),
    examples=[
        "How do I create a custom slash command?",
        "What are hooks and how do I use them?",
        "How do I configure an MCP server?",
        "What is CLAUDE.md?",
        "What permission modes are available?",
    ],
).launch(share=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

---
## Recap: What We Built

```
┌────────────────────────────────────────────────────────────────────┐
│                    Our RAG Pipeline                                │
│                                                                    │
│  1. LOAD        Web docs (httpx) + PDF (LangChain PyPDFLoader)     │
│       │                                                            │
│  2. CHUNK       LangChain RecursiveCharacterTextSplitter           │
│       │         (1500 chars, 200 overlap)                          │
│       │                                                            │
│  3. EMBED       OpenAI text-embedding-3-small → ChromaDB           │
│       │         (1536-dim vectors, cosine similarity)              │
│       │                                                            │
│  4. RETRIEVE    Chroma vector search (top-5 chunks)                │
│       │                                                            │
│  5. GENERATE    gpt-4o-mini with grounded system prompt            │
│       │                                                            │
│  6. EVAL        Retrieval hit-rate + LLM-as-judge (1-5 score)      │
└────────────────────────────────────────────────────────────────────┘
```

### What we used vs. what the original notebook used

| Step | Original notebook | This notebook |
|---|---|---|
| Document loading | Custom regex parser | LangChain `Document` objects |
| PDF support | None | LangChain `PyPDFLoader` |
| Chunking | Hand-written `split_by_size()` | LangChain `RecursiveCharacterTextSplitter` |
| Everything else | Same | Same (OpenAI, Chroma, Gradio) |

### Known limitations of this "naive" RAG

This works surprisingly well for a first pass, but it has real weaknesses:

| Weakness | Example | Fix (advanced topic) |
|---|---|---|
| **Keyword misses** | Searching for an exact error message or flag name may not match semantically | Hybrid search (combine vector + keyword/BM25) |
| **Lost context** | A chunk from the middle of a page doesn't know what page it's on | Contextual chunking (prepend page summary to each chunk) |
| **Multi-hop questions** | "How do I do X given constraint Y?" needs info from 2 different pages | Query rewriting / decomposition |
| **No conversation memory** | Follow-up questions lose context of previous turns | Add chat history to the prompt |